In [ ]:
!pip install mysql-connector-python pandas tqdm

In [ ]:
import pandas as pd
import mysql.connector
from tqdm import tqdm

In [ ]:
import getpass
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password=getpass.getpass("MySQL password: "),
    database="ecommerce_sales"
)

cursor = conn.cursor()

print("✅ Connected Successfully!")

In [ ]:
csv_path = r"C:\Users\ANANYA\OneDrive\Documents\Ecommerce_Customer_Sales_Analytics\Dataset\Cleaned\online_retail_cleaned.csv"

df = pd.read_csv(
    csv_path,
    low_memory=False,
    dtype={"Invoice": str}
)

print(df.shape)
df.head()

In [ ]:
# Convert InvoiceDate to MySQL DATETIME format
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

df["InvoiceDate"] = df["InvoiceDate"].dt.strftime("%Y-%m-%d %H:%M:%S")

# Replace missing values with None
df = df.where(pd.notnull(df), None)

print("Data prepared successfully!")

In [ ]:
cursor.execute("TRUNCATE TABLE online_retail")
conn.commit()

print("Table cleared!")

In [ ]:
query = """
INSERT INTO online_retail
(
    Invoice,
    StockCode,
    Description,
    Quantity,
    InvoiceDate,
    Price,
    CustomerID,
    Country,
    Revenue
)
VALUES
(
    %s,%s,%s,%s,%s,%s,%s,%s,%s
)
"""

data = list(df.itertuples(index=False, name=None))

batch_size = 5000

for i in tqdm(range(0, len(data), batch_size)):
    batch = data[i:i + batch_size]
    cursor.executemany(query, batch)
    conn.commit()

print("✅ Import Completed Successfully!")

In [ ]:
cursor.execute("SELECT COUNT(*) FROM online_retail")

rows = cursor.fetchone()

print("Total Rows Imported:", rows[0])

In [ ]:
cursor.execute("SELECT * FROM online_retail LIMIT 5")

for row in cursor.fetchall():
    print(row)

In [ ]:
cursor.close()
conn.close()

print("Connection Closed!")

In [ ]:
import getpass
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password=getpass.getpass("MySQL password: "),
    database="ecommerce_sales"
)

cursor = conn.cursor()

print("Connected:", conn.is_connected())

In [ ]:
cursor.execute("""
INSERT INTO online_retail
(Invoice, StockCode, Description, Quantity, InvoiceDate, Price, CustomerID, Country, Revenue)
VALUES
('TEST001','ABC123','Test Product',1,'2024-01-01 10:00:00',10.50,1,'India',10.50)
""")

conn.commit()

cursor.execute("SELECT COUNT(*) FROM online_retail")
print(cursor.fetchone())

In [ ]:
print(df.dtypes)
print(df.isnull().sum())
print(df.head())

In [ ]:
import getpass
import pandas as pd
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password=getpass.getpass("MySQL password: "),
    database="ecommerce_sales"
)

cursor = conn.cursor()

# Read CSV
df = pd.read_csv(
    r"C:\Users\ANANYA\OneDrive\Documents\Ecommerce_Customer_Sales_Analytics\Dataset\Cleaned\online_retail_sql.csv",
    low_memory=False
)

# Clean data
df["Customer ID"] = df["Customer ID"].fillna(0).astype(int)
df["Description"] = df["Description"].fillna("")
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"]).dt.strftime("%Y-%m-%d %H:%M:%S")

# Clear table
cursor.execute("TRUNCATE TABLE online_retail")

query = """
INSERT INTO online_retail
(Invoice, StockCode, Description, Quantity, InvoiceDate,
 Price, CustomerID, Country, Revenue)
VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)
"""

data = list(zip(
    df["Invoice"].astype(str),
    df["StockCode"].astype(str),
    df["Description"],
    df["Quantity"].astype(int),
    df["InvoiceDate"],
    df["Price"].astype(float),
    df["Customer ID"].astype(int),
    df["Country"],
    df["Revenue"].astype(float)
))

cursor.executemany(query, data)
conn.commit()

cursor.execute("SELECT COUNT(*) FROM online_retail")
print("Rows Imported:", cursor.fetchone()[0])

cursor.close()
conn.close()